# LSM Tradutor — treino no Google Colab

O servidor já treina o modelo sozinho (botão **Treinar modelo**, demora segundos). Use este notebook quando:

* juntar dados gravados por **várias pessoas / computadores** (vários ZIPs exportados);
* quiser **testar vários algoritmos** e escolher o melhor com validação cruzada mais rigorosa.

**Passos:** na app, aba *Treinar sinais* → **Exportar dados** (gera `lsm-dataset.zip`). Carregue aqui um ou mais ZIPs, execute todas as células e descarregue `model.joblib`.
Depois copie-o para a pasta `data/` do projeto e reinicie: `docker compose restart backend`.

In [ ]:
!pip -q install scikit-learn==1.6.0 numpy==2.2.1 joblib==1.4.2  # mesmas versões do servidor
# depois desta célula: Runtime > Restart session, e continue na célula seguinte

In [ ]:
from google.colab import files
uploaded = files.upload()  # escolha um ou mais lsm-dataset*.zip

In [ ]:
import io, json, zipfile
samples = []
for name, data in uploaded.items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for n in z.namelist():
            if n.startswith('samples/') and n.endswith('.json'):
                s = json.loads(z.read(n))
                s['source'] = name  # quem gravou (para validação por pessoa)
                samples.append(s)
print(len(samples), 'amostras')
from collections import Counter
Counter(s['label'] for s in samples).most_common()

## Extração de características (idêntica ao servidor — `backend/app/features.py`)

In [ ]:
"""
Extração de características a partir dos pontos das mãos (MediaPipe Hand Landmarker).

Cada amostra é uma sequência de frames. Cada frame tem 0, 1 ou 2 mãos, e cada mão
tem 21 pontos (x, y, z) normalizados à imagem.

O MESMO código é usado no servidor e no notebook do Google Colab, para que um modelo
treinado num lado funcione no outro.
"""
from __future__ import annotations

import numpy as np

N_POINTS = 21
T = 16                      # nº de frames para onde cada sinal é reamostrado
HAND_DIM = N_POINTS * 3 + 3 # pontos relativos + posição do pulso (x, y) + presença
FRAME_DIM = HAND_DIM * 2


FEATURE_VERSION = 2  # mude quando a extração mudar (obriga a retreinar)


def _hand_vector(hand: dict | None, mirror: bool = False) -> np.ndarray:
    if not hand:
        return np.zeros(HAND_DIM, dtype=np.float32)
    pts = np.asarray(hand["landmarks"], dtype=np.float32).reshape(N_POINTS, 3)
    wrist = pts[0].copy()
    rel = pts - wrist
    scale = float(np.max(np.linalg.norm(rel[:, :2], axis=1))) + 1e-6
    rel /= scale  # invariante ao tamanho da mão / distância à câmara
    wx = wrist[0] - 0.5
    if mirror:  # espelha: canhotos e destros produzem o mesmo vetor
        rel[:, 0] *= -1
        wx = -wx
    extra = np.array([wx, wrist[1] - 0.5, 1.0], dtype=np.float32)
    return np.concatenate([rel.ravel(), extra])


def frame_vector(frame: dict) -> np.ndarray:
    hands = frame.get("hands") or []
    if len(hands) == 1:
        h = hands[0]
        # uma mão: vai sempre para o 1º lugar, espelhada se for a esquerda
        return np.concatenate([
            _hand_vector(h, mirror=h.get("handedness") == "Left"),
            np.zeros(HAND_DIM, dtype=np.float32),
        ])
    if len(hands) >= 2:
        # duas mãos: ordena pela posição na imagem (mais fiável que o rótulo esquerda/direita)
        a, b = sorted(hands[:2], key=lambda h: h["landmarks"][0][0])
        return np.concatenate([_hand_vector(a), _hand_vector(b, mirror=True)])
    return np.zeros(FRAME_DIM, dtype=np.float32)


def has_hands(frame: dict) -> bool:
    return bool(frame.get("hands"))


def trim(frames: list[dict]) -> list[dict]:
    """Remove frames sem mãos no início e no fim."""
    idx = [i for i, f in enumerate(frames) if has_hands(f)]
    if not idx:
        return []
    return frames[idx[0]: idx[-1] + 1]


def resample(seq: np.ndarray, t: int = T) -> np.ndarray:
    n = len(seq)
    if n == 1:
        return np.repeat(seq, t, axis=0)
    src = np.linspace(0, n - 1, t)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, n - 1)
    w = (src - lo)[:, None]
    return seq[lo] * (1 - w) + seq[hi] * w


def sequence_features(frames: list[dict]) -> np.ndarray | None:
    frames = trim(frames)
    if sum(has_hands(f) for f in frames) < 3:
        return None
    seq = np.stack([frame_vector(f) for f in frames])
    return seq_to_features(seq)


def seq_to_features(seq: np.ndarray) -> np.ndarray:
    seq = resample(seq)
    motion = np.abs(np.diff(seq, axis=0)).mean(axis=0)  # quanto cada ponto se move
    return np.concatenate([seq.ravel(), motion * 10]).astype(np.float32)


def augment(seq: np.ndarray, rng: np.random.Generator, n: int = 6) -> list[np.ndarray]:
    """Gera variações de uma sequência (ruído, escala, velocidade) para treinar melhor
    com poucas amostras."""
    out = []
    for _ in range(n):
        s = seq.copy()
        s += rng.normal(0, 0.02, s.shape).astype(np.float32)
        s *= rng.uniform(0.92, 1.08)
        # deformação temporal: corta/estica ligeiramente
        k = len(s)
        if k > 4:
            a = rng.integers(0, max(1, k // 8))
            b = k - rng.integers(0, max(1, k // 8))
            s = s[a:b]
        out.append(seq_to_features(s))
    return out


def raw_sequence(frames: list[dict]) -> np.ndarray | None:
    frames = trim(frames)
    if sum(has_hands(f) for f in frames) < 3:
        return None
    return np.stack([frame_vector(f) for f in frames])


In [ ]:
import numpy as np
seqs, labels, groups = [], [], []
for s in samples:
    q = raw_sequence(s['frames'])
    if q is not None:
        seqs.append(q); labels.append(s['label']); groups.append(s['source'])
counts = Counter(labels)
keep = {l for l, c in counts.items() if c >= 3}
idx = [i for i, l in enumerate(labels) if l in keep]
seqs = [seqs[i] for i in idx]; labels = np.array([labels[i] for i in idx]); groups = np.array([groups[i] for i in idx])
print(len(seqs), 'amostras válidas,', len(keep), 'sinais')

rng = np.random.default_rng(0)
def build(ix, aug=True):
    X, y = [], []
    for i in ix:
        X.append(seq_to_features(seqs[i])); y.append(labels[i])
        if aug:
            for f in augment(seqs[i], rng):
                X.append(f); y.append(labels[i])
    return np.stack(X), np.array(y)

## Comparar modelos com validação cruzada

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold

candidates = {
    'ExtraTrees': lambda: ExtraTreesClassifier(n_estimators=400, max_features='sqrt', n_jobs=-1, class_weight='balanced', random_state=42),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=400, n_jobs=-1, class_weight='balanced', random_state=42),
    'MLP': lambda: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=400, early_stopping=True, random_state=42)),
}

n_people = len(set(groups))
if n_people >= 2:
    print(f'Validação por pessoa/ficheiro ({n_people} grupos) — mede se generaliza para quem não gravou')
    splits = list(GroupKFold(n_splits=min(5, n_people)).split(np.zeros(len(labels)), labels, groups))
else:
    k = min(5, min(Counter(labels).values()))
    splits = list(StratifiedKFold(n_splits=k, shuffle=True, random_state=0).split(np.zeros(len(labels)), labels))

results = {}
for name, make in candidates.items():
    accs = []
    for tr, te in splits:
        Xtr, ytr = build(tr)
        Xte = np.stack([seq_to_features(seqs[i]) for i in te])
        clf = make().fit(Xtr, ytr)
        accs.append((clf.predict(Xte) == labels[te]).mean())
    results[name] = float(np.mean(accs))
    print(f'{name:12s} precisão média: {results[name]:.1%}')
best = max(results, key=results.get)
print('Melhor:', best)

## Treinar o melhor modelo com todos os dados e descarregar

In [ ]:
import joblib, time
X, y = build(range(len(seqs)))
clf = candidates[best]().fit(X, y)
meta = {
    'classes': sorted(keep), 'n_samples': len(seqs), 'accuracy': results[best],
    'skipped': sorted(l for l in counts if l not in keep),
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S') + ' (Colab, ' + best + ')', 'seconds': 0, 'feature_version': FEATURE_VERSION,
}
joblib.dump({'clf': clf, 'meta': meta}, 'model.joblib')
files.download('model.joblib')

### Matriz de confusão (que sinais o modelo confunde)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
tr, te = splits[0]
m = candidates[best]().fit(*build(tr))
Xte = np.stack([seq_to_features(seqs[i]) for i in te])
fig, ax = plt.subplots(figsize=(14, 14))
ConfusionMatrixDisplay.from_predictions(labels[te], m.predict(Xte), ax=ax, xticks_rotation=90, colorbar=False)
plt.tight_layout()